# DeepVariant + GLnexus SNV/indel stats (merge)

Merge chromosome-level `bcftools stats` files written back to
`GL_INTERVAL_set` by `BcftoolsGlnexusStats.wdl`.

The table lives in the storage workspace
[`allofus-drc-wgs-LR-prodData` / `AoU_DRC_LongReads_PhaseTwo_Storage`](https://app.terra.bio/#workspaces/allofus-drc-wgs-LR-prodData/AoU_DRC_LongReads_PhaseTwo_Storage).
This notebook pulls it with firecloud, downloads each nuclear row's `stats`
URI, and merges site counts plus per-sample Ti/Tv and het/hom.

`chrM` is listed on the table but is skipped for the nuclear catalog
(chr1–22, X, Y). Empty stats (0 records) fail the merge — that was the
`-f PASS` / `FILTER=.` mismatch.

Indel length bins **<20 bp** and **<50 bp** are available for the **site
catalog** (bcftools IDD). Per-sample `n_indel` is not length-binned.

Per-sample means are reported for all VCF samples, non-control samples
(IDs not starting with `HG` or `NA`), and Phase 2 PacBio discovery if
covariates are on the VM. Nuclear means feed the manuscript
“SNVs and indels per participant” sentence.

**Terra:** `edit/scripts/` on the VM is often stale. From a current git
checkout, stage CLIs to the workspace bucket, then re-run the setup cell:

```bash
gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/"
```

Set `SNV_RUN_PIPELINE=true` to download stats and merge. Override the table
with `SNV_TABLE_TSV` only if you already exported `GL_INTERVAL_set`.
Re-merge does not need `SNV_FORCE` if the shard files are already local.

## Outputs

- `summaries/manuscript/snv_indel_site_counts.{tsv,md}`
- `summaries/manuscript/snv_indel_sample_qc.tsv`
- `summaries/manuscript/snv_indel_sample_qc_summary.{tsv,md}`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# On Terra, localize $WORKSPACE_BUCKET/scripts/ before importing anything.
# Persistent edit/scripts/ copies are often stale and must not win.
_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file() or (_d / "workspace_paths.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
    os.environ["TERRA_SCRIPTS_LOCALIZED"] = "true"
    import importlib
    importlib.invalidate_caches()
    _prefix = str(_scripts)
    for _name, _mod in list(sys.modules.items()):
        _file = getattr(_mod, "__file__", None)
        if _file and str(_file).startswith(_prefix):
            sys.modules.pop(_name, None)
elif _scripts is None:
    raise FileNotFoundError(
        "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
        "Upload scripts/ to gs://WORKSPACE/scripts/."
    )
sys.path.insert(0, str(_scripts))

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py",
    "resolve_gl_interval_manifest.py",
    "snv_bcftools_sample_qc.py",
)
from workspace_paths import data_root
try:
    from workspace_paths import snv_output_dir
except ImportError as exc:
    raise ImportError(
        f"{getattr(sys.modules.get('workspace_paths'), '__file__', 'workspace_paths')} is stale "
        "(no snv_output_dir). From a current git checkout run "
        'gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/" '
        "and re-run this cell."
    ) from exc
from resolve_gl_interval_manifest import (
    DEFAULT_ENTITY_TYPE as GL_INTERVAL_ENTITY_TYPE,
    DEFAULT_NAMESPACE as TERRA_NAMESPACE_DEFAULT,
    DEFAULT_WORKSPACE as TERRA_WORKSPACE_DEFAULT,
    fetch_gl_interval_manifest_firecloud,
    load_gl_interval_manifest_tsv,
    natural_chrom_key,
    normalize_gl_interval_manifest,
)
from snv_bcftools_sample_qc import (
    env_flag,
    hg_na_mask,
    manuscript_per_participant_sentence,
    merge_shards,
    pull_stats_from_table,
    select_merge_intervals,
    stats_uri_column,
    write_outputs,
)

import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = data_root()
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
SUMMARY_DIR = Path(os.environ.get("SNV_SUMMARY_DIR", ROOT / "summaries" / "manuscript"))
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR = Path(os.environ.get("SNV_SHARD_DIR", snv_output_dir() / "shards"))
STATS_DIR.mkdir(parents=True, exist_ok=True)
TABLE_TSV = os.environ.get("SNV_TABLE_TSV", "")
COV_CSV = Path(
    os.environ.get(
        "AOU_COVARIATES",
        ROOT / "covariates.source_rebuilt.csv.gz",
    )
)
if not COV_CSV.is_file():
    alt = Path.cwd() / "covariates.v6.csv.gz"
    if alt.is_file():
        COV_CSV = alt

RUN_PIPELINE = env_flag("SNV_RUN_PIPELINE")
FORCE = env_flag("SNV_FORCE")
AUTOSOMES_ONLY = env_flag("SNV_AUTOSOMES_ONLY", default=False)
TERRA_NAMESPACE = os.environ.get("SNV_TERRA_NAMESPACE", TERRA_NAMESPACE_DEFAULT)
TERRA_WORKSPACE = os.environ.get("SNV_TERRA_WORKSPACE", TERRA_WORKSPACE_DEFAULT)
ENTITY_TYPE = os.environ.get("SNV_TERRA_ENTITY_TYPE", GL_INTERVAL_ENTITY_TYPE)
PULL_JOBS = int(os.environ.get("SNV_PULL_JOBS", "8"))

print("ROOT:", ROOT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET or "(local)")
print("SUMMARY_DIR:", SUMMARY_DIR)
print("STATS_DIR:", STATS_DIR)
print("COV_CSV:", COV_CSV, "exists=" + str(COV_CSV.is_file()))
print("RUN_PIPELINE:", RUN_PIPELINE)
print("TERRA:", f"{TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}")
print("PULL_JOBS:", PULL_JOBS)
print("AUTOSOMES_ONLY:", AUTOSOMES_ONLY)
print("FORCE:", FORCE)


## 1. Fetch `GL_INTERVAL_set`

Uses firecloud against `AoU_DRC_LongReads_PhaseTwo_Storage` unless
`SNV_TABLE_TSV` points at an export. This cell does not need
`SNV_RUN_PIPELINE`.


In [ ]:
table_tsv_out = SUMMARY_DIR / "gl_interval_set.table.tsv"

if TABLE_TSV:
    print("table TSV:", TABLE_TSV)
    raw = load_gl_interval_manifest_tsv(Path(TABLE_TSV))
    source_label = TABLE_TSV
else:
    print(f"firecloud get_entities {TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}")
    raw = fetch_gl_interval_manifest_firecloud(
        TERRA_NAMESPACE, TERRA_WORKSPACE, ENTITY_TYPE
    )
    source_label = f"{TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}"

chrom_manifest = normalize_gl_interval_manifest(
    raw, autosomes_only=False, source_label=source_label
)
chrom_manifest = chrom_manifest.sort_values(
    "interval_id", key=lambda s: s.map(natural_chrom_key)
).reset_index(drop=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
chrom_manifest.to_csv(table_tsv_out, sep="\t", index=False)

try:
    stats_col = stats_uri_column(chrom_manifest)
except ValueError:
    stats_col = None
n_stats = (
    int(chrom_manifest[stats_col].astype(str).str.startswith("gs://").sum())
    if stats_col
    else 0
)
print(f"rows={len(chrom_manifest):,}  stats column={stats_col or '(none)'}  gs://={n_stats}")
print("intervals:", ", ".join(chrom_manifest["interval_id"]))
cols = ["interval_id", "VCF"] + ([stats_col] if stats_col else [])
display(chrom_manifest[cols])


## 2. Pull `stats` and merge nuclear shards

Downloads `{chrN}.stats.txt` from the table, skips `chrM`, and requires
chr1–22 / X / Y with a non-zero `number of records`. Set
`SNV_RUN_PIPELINE=true`.


In [ ]:
n_local = len(list(STATS_DIR.glob("*.stats.txt"))) if STATS_DIR.is_dir() else 0
print("local stats:", n_local, "in", STATS_DIR)
print("table stats URIs:", n_stats, "column=", stats_col or "(none)")

if not RUN_PIPELINE:
    print("dry-run: set SNV_RUN_PIPELINE=true to pull stats and merge")
else:
    if stats_col and n_stats:
        pull = pull_stats_from_table(
            chrom_manifest,
            STATS_DIR,
            jobs=PULL_JOBS,
            force=FORCE,
        )
        display(pull)
    elif n_local == 0:
        raise FileNotFoundError(
            "No stats URIs on GL_INTERVAL_set and none in STATS_DIR. "
            "Submit BcftoolsGlnexusStats.wdl with outputs written back to the table."
        )
    intervals = select_merge_intervals(
        chrom_manifest["interval_id"].tolist(),
        autosomes_only=AUTOSOMES_ONLY,
    )
    print("merge intervals:", ", ".join(intervals))
    catalog, sample_df = merge_shards(STATS_DIR, intervals)
    paths = write_outputs(catalog, sample_df, SUMMARY_DIR, covariates=COV_CSV if COV_CSV.is_file() else None)
    display(catalog.T)
    summary = pd.read_csv(paths["summary_tsv"], sep="\t")
    display(summary)
    n_ctrl = int(hg_na_mask(sample_df["research_id"]).sum())
    print(f"HG/NA controls in VCF: {n_ctrl:,} / {len(sample_df):,}")
    print(manuscript_per_participant_sentence(summary))
    display(sample_df.head())
